# Does the way we ask gpt-oss change which word it guesses?

The spymaster's guesser model (the *listener*) is trained on how gpt-oss-120b
guesses. Every training example was bought with one prompt: show the board and
the clue, and ask gpt-oss to **rank all of the words**. The first word of that
ranking is read as the first guess, the second word as the second guess, and so on.

That reading is a choice. A real guesser is asked one guess at a time, and
learns before its second guess that the first one was right. This notebook asks
whether gpt-oss picks differently when it is asked those other ways.

**Setup**

- **300 held-out positions**: a board plus a clue and number, taken from boards
  no listener was trained on.
- **Four prompts** (below), each asked **8 times per position** with the
  temperature set to 1.0, so each prompt gives a small sample of picks.
  Temperature has to be set explicitly: at the provider's default, repeated
  identical requests come back word-for-word identical, and there would be no
  distribution to compare.
- **Model**: `openai/gpt-oss-120b` via DeepInfra.

Everything below reads the stored answers in `cache/prompt_variants.db`. No
API calls are made, and re-running the notebook reproduces the same numbers.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts" / "tools")]

# The analysis script's own functions, so these tables are the same
# computation as scripts/tools/compare_prompt_methods.py.
from compare_prompt_methods import METHODS, VARIANTS, boot, load, tv, tv_with_null

pos = load(VARIANTS)
print(f"{len(pos)} positions, {sum('first' in p for p in pos.values())} with a second step")

300 positions, 225 with a second step


## The four prompts

| Prompt | What gpt-oss is asked | Step 2 (the guess after the first) |
|---|---|---|
| `ranked` | Rank **all** the words for the clue and number. *The training prompt.* | The 2nd word of its ranking |
| `ranked_nonumber` | The same, without telling it the number | The 2nd word of its ranking |
| `single` | Name the **one** word you would guess first | Asked again with the first word silently removed from the board |
| `single_feedback` | *(step 2 only)* | The first word removed, **and** told: "You already guessed X, and it was correct" |

**How step 2 is made comparable.** Each position has one fixed first word: the
first word of the ranking in the training data. The `single` prompts remove that
word from the board. For the `ranked` prompts, only the samples whose own first
word is that same word are kept, and their second word is used. So the ranked
prompts have fewer than 8 step-2 picks at some positions.

The exact prompt text:

In [2]:
from codenames.guessers.llm import _PROMPT_TEMPLATE
sys.path.insert(0, str(ROOT / "scripts" / "data"))
from collect_prompt_variants import SINGLE, SINGLE_FEEDBACK

for name, text in [("ranked (training prompt)", _PROMPT_TEMPLATE),
                   ("single", SINGLE), ("single_feedback", SINGLE_FEEDBACK)]:
    print(f"===== {name}\n{text}\n")

===== ranked (training prompt)
You are playing the guesser role in the board game Codenames. Your spymaster gave the clue "{clue}"{count_note}. Here are the words still available to guess:

{words}

Rank ALL of the words above by how likely you would guess them for this clue, most likely first. Respond with ONLY a JSON array of the words as strings, e.g. ["word1", "word2", ...] -- no other text. Every word listed above must appear exactly once in your answer.

===== single
You are playing the guesser role in the board game Codenames. Your spymaster gave the clue "{clue}"{count_note}. Here are the words still available to guess:

{words}

Which ONE of the words above would you guess first? Respond with ONLY a JSON array containing that single word as a string, e.g. ["word"] -- no other text.

===== single_feedback
You are playing the guesser role in the board game Codenames. Your spymaster gave the clue "{clue}"{count_note}. You already guessed "{prev}", and it was correct -- it was one

## How a difference between two prompts is measured

**Total variation (TV).** At one position, turn each prompt's 8 picks into a
frequency distribution *p* and *q* over the board words, then

$$\mathrm{TV}(p, q) = \tfrac{1}{2} \sum_{w} \lvert p(w) - q(w) \rvert$$

TV is the share of probability that would have to move to turn one distribution
into the other: 0 means identical, 1 means no word in common.

*Example.* Prompt A picks Plot 6 times and Tail 2 times; prompt B picks Plot 4,
Tail 2 and Book 2. Then TV = ½ (|0.75 − 0.50| + |0.25 − 0.25| + |0 − 0.25|) = 0.25.

**Removing sampling noise.** With only 8 picks a side, two samples from the
*same* prompt still differ, so raw TV is inflated. At each position we:

1. pool the two prompts' picks,
2. shuffle them randomly into two groups of the original sizes and compute TV,
3. repeat 200 times.

The average is the **noise**: the TV expected if the prompts made no difference.
**Excess = TV − noise** is the part of the difference the prompt causes.
Shuffling only within a position matters, because each position has its own
distribution.

**Summaries across positions.**

- *Excess* is averaged over positions, with a 95% bootstrap interval that
  resamples positions.
- *Same top word* is the share of positions where both prompts' most frequent
  pick is the same word.
- *Share p < 0.05* is the share of positions where the shuffle test on its own
  rejects "no difference". If the prompts never differed, it would sit near 5%.
  With 8 picks a side, a single position rarely has the data to pass, so this
  column understates real differences.

All tables are computed in the next cell, in one pass with a fixed random seed.

In [3]:
def shape_table(step):
    rows = []
    for m in METHODS[step]:
        share, distinct, ent = [], [], []
        for p in pos.values():
            v = p.get((m, step), [])
            if len(v) < 2:
                continue
            c = np.array(list(Counter(v).values()), dtype=float) / len(v)
            share.append(c.max())
            distinct.append(len(c))
            ent.append(float(-(c * np.log2(c)).sum()))
        rows.append({"prompt": m, "top word's share": np.mean(share),
                     "distinct words (of 8)": np.mean(distinct), "entropy (bits)": np.mean(ent)})
    return pd.DataFrame(rows).set_index("prompt").round(3)


def tv_table(step, rng):
    ms, rows = METHODS[step], []
    for i, a in enumerate(ms):
        for b in ms[i + 1:]:
            obs, null, sig, mode = [], [], [], []
            for p in pos.values():
                va, vb = p.get((a, step), []), p.get((b, step), [])
                if len(va) < 2 or len(vb) < 2:
                    continue
                o, n, pv = tv_with_null(va, vb, rng)
                obs.append(o); null.append(n); sig.append(pv < 0.05)
                mode.append(Counter(va).most_common(1)[0][0] == Counter(vb).most_common(1)[0][0])
            ex = np.array(obs) - np.array(null)
            lo, hi = boot(ex)
            rows.append({"pair": f"{a} vs {b}", "TV": np.mean(obs), "noise": np.mean(null),
                         "excess": ex.mean(), "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                         "same top word": f"{np.mean(mode):.0%}", "share p < 0.05": f"{np.mean(sig):.1%}",
                         "positions": len(obs)})
    return pd.DataFrame(rows).set_index("pair").round(3)


rng = np.random.default_rng(0)       # same seed and order as the script, so the numbers match it
shape1, tv1 = shape_table(1), tv_table(1, rng)
shape2, tv2 = shape_table(2), tv_table(2, rng)

## Step 1: the first guess (300 positions)

In [4]:
tv1

,TV,noise,excess,95% CI,same top word,share p < 0.05,positions
pair,,,,,,,
ranked vs ranked_nonumber,0.129,0.120,0.009,"[-0.002, +0.020]",89%,1.7%,300
ranked vs single,0.148,0.123,0.025,"[+0.013, +0.038]",87%,2.7%,300
ranked_nonumber vs single,0.151,0.121,0.031,"[+0.018, +0.043]",88%,2.0%,300


In [5]:
shape1

,top word's share,distinct words (of 8),entropy (bits)
prompt,,,
ranked,0.854,1.733,0.473
ranked_nonumber,0.849,1.730,0.475
single,0.852,1.690,0.460


**Reading step 1.** The first guess barely depends on the prompt. At most
0.03 of probability moves between any two prompts, and they agree on the top
word at about 88% of positions. The two ranked prompts are indistinguishable,
so telling gpt-oss the number changes nothing. The single-word prompt differs
from them by a small but detectable amount (excess about +0.03, interval above
zero). All three distributions have the same shape: the top word takes about
85% of the picks.

## Step 2: the guess after the first

In [6]:
tv2

,TV,noise,excess,95% CI,same top word,share p < 0.05,positions
pair,,,,,,,
ranked vs ranked_nonumber,0.214,0.208,0.007,"[-0.010, +0.024]",78%,1.0%,199
ranked vs single,0.329,0.242,0.087,"[+0.065, +0.110]",66%,8.7%,206
ranked vs single_feedback,0.327,0.239,0.089,"[+0.065, +0.112]",67%,10.2%,206
ranked_nonumber vs single,0.355,0.243,0.112,"[+0.087, +0.137]",63%,14.1%,205
ranked_nonumber vs single_feedback,0.333,0.240,0.093,"[+0.070, +0.117]",64%,10.2%,205
single vs single_feedback,0.286,0.236,0.050,"[+0.033, +0.069]",70%,4.9%,225


In [7]:
shape2

,top word's share,distinct words (of 8),entropy (bits)
prompt,,,
ranked,0.763,2.068,0.725
ranked_nonumber,0.775,2.049,0.701
single,0.717,2.516,0.935
single_feedback,0.714,2.462,0.926


**Reading step 2.** Here the prompts split into two families.

- **List continuation** (`ranked`, `ranked_nonumber`): the two agree with each
  other (excess +0.007, interval covering zero).
- **Re-asking** (`single`, `single_feedback`): differs from list continuation by
  about 9–11% of probability, with every interval well above zero. The two
  families agree on the top word at only about 65% of positions. Re-asks are
  also more spread out (entropy about 0.93 bits against 0.72).
- **Saying the first guess was correct** moves the re-ask by a smaller amount
  (excess +0.05).

So the step-2 pick in our training data (the second word of a ranked list) is
not the same as what gpt-oss picks when it is asked for its next guess.

## Examples: where list continuation and re-asking disagree most

Each row is one position, with the counts of each pick out of the samples.

In [8]:
pd.set_option("display.max_colwidth", None)
fmt = lambda v: ", ".join(f"{w} {c}" for w, c in Counter(v).most_common())
scored = []
for (seed, clue, number), p in pos.items():
    va, vb = p.get(("ranked", 2), []), p.get(("single", 2), [])
    if len(va) >= 4 and len(vb) >= 4:
        scored.append((tv(va, vb), clue, number, p))
scored.sort(key=lambda t: -t[0])

pd.DataFrame([{
    "clue": f"{clue.upper()} {number}", "first guess (removed)": p["first"],
    "step 2, ranked (continue the list)": fmt(p[("ranked", 2)]),
    "step 2, single (re-ask)": fmt(p[("single", 2)]),
    "step 2, single_feedback": fmt(p[("single_feedback", 2)]),
    "TV": round(d, 2)} for d, clue, number, p in scored[:8]]).set_index("clue")

,first guess (removed),"step 2, ranked (continue the list)","step 2, single (re-ask)","step 2, single_feedback",TV
clue,,,,,
EXPLORING 3,Himalayas,Africa 8,"Mercury 5, Saturn 2, Helicopter 1","Saturn 6, Mercury 2",1.00
CONSISTED 4,Part,"Contract 7, Pumpkin 1","Europe 2, Rome 1, Ice cream 1, Pound 1, Mammoth 1, Cook 1, Pie 1","Europe 3, Contract 3, Pie 1, Pumpkin 1",1.00
ATLANTA 2,Phoenix,Tokyo 6,"Pie 3, Greece 2, Stock 1, Pitch 1, Pumpkin 1","Tokyo 4, Greece 2, Pie 1, Pitch 1",1.00
UNFORTUNATELY 4,Luck,"Slug 3, Hood 1, Well 1, Night 1, Engine 1","Dice 3, Turkey 2, Night 2, Cover 1","Dice 3, Night 2, Mercury 1, Turkey 1, Well 1",0.86
PLAZA 2,London,"Embassy 5, Stadium 1",Stadium 8,"Embassy 5, Stadium 3",0.83
COMPOSER 2,Conductor,"Fighter 6, Brush 1, Orange 1","Brush 4, Mole 1, Orange 1, Card 1, Marble 1","Brush 3, Fighter 3, Orange 2",0.75
MIGHT 4,Force,Boom 8,"Field 6, Boom 2","Pole 4, Field 2, Block 1, Boom 1",0.75
INSPIRED 2,Genius,"Scientist 7, Fire 1","Fire 3, Olympus 2, Scientist 1, Figure 1, Jam 1",Scientist 8,0.75


These are the 8 most extreme positions, chosen for the size of the
disagreement, so they illustrate the kinds of difference rather than a typical
position. Two patterns recur:

- **A change of word sense.** EXPLORING: having named the Himalayas, the list
  continues on Earth (Africa), while the re-ask turns to space (Saturn, Mercury).
  PLAZA: the list stays with Embassy while the re-ask picks Stadium. ATLANTA: the
  list continues with another city (Tokyo), while the re-ask drifts to Greece and
  Pie.
- **Scatter when nothing fits well.** CONSISTED, UNFORTUNATELY, COMPOSER: the list
  commits to one word, while the re-ask spreads over four or five, which is
  where the higher step-2 entropy comes from.

Telling gpt-oss that the first guess was correct (`single_feedback`) sometimes
pulls the re-ask back toward the list's answer (ATLANTA: Tokyo; PLAZA: Embassy;
INSPIRED: Scientist), consistent with it sitting between the two families.

## Caveats

- **8 picks per position is enough for the averages above, but not for any single
  board.** The noise level (0.12–0.24) is as large as the effects, which is why few
  positions pass on their own. Per-board conclusions would need more samples
  (e.g. 32).
- **Excess TV probably understates real differences.** The shuffle measures the
  noise inflation as if the prompts were identical; when they truly differ, the
  inflation is smaller, so subtracting it removes slightly too much. The step-2
  gap is therefore, if anything, larger than shown.
- **The ranked prompts' step-2 rows are thinner.** They use only samples that began
  with the fixed first word.
- **The training data was bought at the provider's default sampling**, which is
  close to always taking the most likely word. The samples here use temperature
  1.0, so they show the spread of the model's preferences rather than its single
  most likely answer.